# CS 521 HW3, Problem 1: PGD Adversarial Training

# Set up for dataset and model

In [1]:
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=27d323fb-2e16-44b4-9002-0868d5f50772
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 66.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 18.4MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)

# PGD Attack (from HW1)

In [5]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True)
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        loss = ce_loss(output,labels)
        loss.backward()
        adv_x = adv_x+(eps_step*adv_x.grad.sign())
        delta = torch.clamp(adv_x-x, min=-eps, max=eps)
        adv_x=torch.clamp(x+delta, 0, 1).detach()

    return adv_x

# Evaluation

In [6]:
def test_model(model,eps,k=20,batch_size=250):
    model.eval()
    correct=0
    for i in range(0,len(x_test),batch_size):
        x=x_test[i:i+batch_size]
        y=y_test[i:i+batch_size]
        if eps == 0:
            x_adv=x
        else:
            x_adv=pgd_linf_untargeted(model,x,y,k,eps,eps_step=eps/4)
        with torch.no_grad():
            pred=model(x_adv).argmax(1)
        correct+=(pred == y).sum().item()
    acc=100.0*correct/len(y_test)
    print(f"eps {eps:.4f}  acc {acc:.1f}")
    return acc

# Training (same as hw3 q3 without RAMP branches)

In [7]:
def adv_train(model, loss_fn, epochs=5, lr=0.01, batch_size=128, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt=torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    start=time.time()
    for epoch in range(epochs):
        losses=[]
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:
                continue
            x=augment(x)
            loss=loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Standard and L-infinity Adversarial Training

In [8]:
def standard_loss(model,x,y):
    model.train()
    return F.cross_entropy(model(x),y)


def linf_adv_loss(model,x,y,steps=10):
    eps=EPS["linf"]
    adv_x=pgd_linf_untargeted(model,x,y,k=steps,eps=eps,eps_step=eps/4)
    model.train()
    return F.cross_entropy(model(adv_x),y)

In [9]:
model_std=adv_train(make_model(),standard_loss,epochs=5)
model_linf=adv_train(make_model(),linf_adv_loss,epochs=5)

epoch 1: loss 1.878 (8 s)
epoch 2: loss 1.556 (13 s)
epoch 3: loss 1.399 (18 s)
epoch 4: loss 1.252 (23 s)
epoch 5: loss 1.162 (28 s)
epoch 1: loss 2.195 (62 s)
epoch 2: loss 2.092 (125 s)
epoch 3: loss 2.049 (188 s)
epoch 4: loss 2.022 (252 s)
epoch 5: loss 2.012 (316 s)


In [10]:
eps_l=[0,0.01,0.02,8/255,0.05,0.1]
print("standard model")
for eps in eps_l:
    test_model(model_std,eps)

print("linf adversarially trained model")
for eps in eps_l:
    test_model(model_linf,eps)

standard model
eps 0.0000  acc 57.9
eps 0.0100  acc 15.1
eps 0.0200  acc 1.9
eps 0.0314  acc 0.2
eps 0.0500  acc 0.1
eps 0.1000  acc 0.0
linf adversarially trained model
eps 0.0000  acc 38.4
eps 0.0100  acc 34.1
eps 0.0200  acc 30.2
eps 0.0314  acc 25.5
eps 0.0500  acc 18.1
eps 0.1000  acc 5.9


In [12]:
def test_fgsm(model, eps, batch_size=250):
    model.eval()
    correct=0
    for i in range(0, len(x_test), batch_size):
        x=x_test[i:i + batch_size]
        y=y_test[i:i + batch_size]
        if eps==0:
            x_adv=x
        else:
            x_adv=pgd_linf_untargeted(model,x,y,1,eps,eps_step=eps)
        with torch.no_grad():
            pred=model(x_adv).argmax(1)
        correct+=(pred==y).sum().item()
    acc=100.0*correct/len(y_test)
    print(f"eps {eps:.4f}  acc {acc:.1f}")
    return acc


print("standard model, FGSM")
for eps in eps_l:
    test_fgsm(model_std, eps)

print("linf adversarially trained model, FGSM")
for eps in eps_l:
    test_fgsm(model_linf, eps)

standard model, FGSM
eps 0.0000  acc 57.9
eps 0.0100  acc 20.9
eps 0.0200  acc 8.0
eps 0.0314  acc 2.9
eps 0.0500  acc 0.8
eps 0.1000  acc 1.2
linf adversarially trained model, FGSM
eps 0.0000  acc 38.4
eps 0.0100  acc 34.0
eps 0.0200  acc 30.2
eps 0.0314  acc 26.2
eps 0.0500  acc 19.3
eps 0.1000  acc 10.3
